## Publish a Pointset object

This example shows how to convert pointset data in CSV format into an Evo geoscience object using the Evo Python SDK.

### Requirements

You must have a Seequent account with the Evo entitlement to use this notebook.

The following parameters must be provided:

- The client ID of your Evo application.
- The callback/redirect URL of your Evo application.

To obtain these app credentials, refer to the [Apps and tokens guide](https://developer.seequent.com/docs/guides/getting-started/apps-and-tokens) in the Seequent Developer Portal.

In [ ]:
from evo.notebooks import ServiceManagerWidget

input_path = "sample-data"

# Evo app credentials
client_id = "<your-client-id>"  # Replace with your client ID
redirect_url = "<your-redirect-url>"  # Replace with your redirect URL

manager = await ServiceManagerWidget.with_auth_code(
    redirect_url=redirect_url,
    client_id=client_id,
).login()

### Load the typed PointSet API

In [ ]:
%load_ext evo.widgets

from evo.objects.typed import EpsgCode, PointSet, PointSetData

### Load the input data

The typed PointSet API can build coordinates, attributes, and categorical lookup tables directly from a dataframe.

In [ ]:
import pandas as pd

# Define the input CSV file.
input_file = f"{input_path}/mixed_attributes.csv"

# Load the point data into a pandas dataframe.
input_df = pd.read_csv(input_file)

input_df.head()

### Define object metadata

The typed objects API creates the pointset directly from a dataframe.

Enter values for these parameters:
- `object_name`: The name of the object.
- `object_path`: The file path where the object will be found.
- `object_epsg_code`: (Optional) The EPSG code for the coordinate reference system.
- `object_tags`: (Optional) A dictionary of tags to assign to the object.

In [ ]:
# Define the basic object metadata.
object_name = "Pointset SDK demo"
object_path = "Jupyter"

# Define the coordinate reference system (CRS) for the object.
# Use EpsgCode(...) for EPSG-based CRS values, or set this to None for an unspecified CRS.
object_epsg_code = EpsgCode(32650)

# Optional tags to attach to the published object.
object_tags = {"Source": "Jupyter Notebook", "Evo SDK": "0.1.5"}
object_description = "Pointset created from CSV data using the typed PointSet API."

# Define the full object path in the workspace.
full_obj_path = f"{object_path}/{object_name}.json"

### Prepare the locations dataframe

In [ ]:
# The typed PointSet API expects x, y, z columns for coordinates.
# Any remaining columns are published as point attributes using the dataframe column order.
locations_df = input_df.rename(columns={"X": "x", "Y": "y", "Z": "z"}).copy()

# Cast the mixed attribute columns to the intended pandas dtypes so the SDK infers
# integer, scalar, bool, string, and categorical attribute types correctly.
locations_df["Bool_Attr"] = locations_df["Bool_Attr"].astype("string").str.lower().map({"true": True, "false": False}).astype("bool")
locations_df["String_Attr"] = locations_df["String_Attr"].astype("string")
locations_df["Category_Attr"] = locations_df["Category_Attr"].astype("category")

locations_df.head()

### Create the pointset data

In [ ]:
# Create the data object used by the typed PointSet API.
# The locations dataframe contains both the coordinates and the point attributes.
pointset_data = PointSetData(
    name=object_name,
    description=object_description,
    locations=locations_df,
    coordinate_reference_system=object_epsg_code,
    tags=object_tags,
)

### Create and publish the pointset

In [ ]:
# Create and publish the pointset at the requested workspace path.
# The typed API handles the schema conversion, data upload, and object creation.
pointset = await PointSet.create(manager, pointset_data, path=full_obj_path)

pointset

Success! You now have a new geoscience object in Evo containing your pointset data.

## Summary

In this example, we've completed the following:
* Loaded point data from CSV into a pandas dataframe.
* Renamed the coordinate columns to `x`, `y`, and `z` for the typed PointSet API.
* Cast `Bool_Attr`, `String_Attr`, and `Category_Attr` to the intended pandas dtypes so the SDK infers the correct attribute types.
* Used the dataframe column order directly when publishing the point attributes.
* Created and published the pointset directly with `PointSet.create(...)`.